# Download the lab-tested genomes straight to Google Drive

Runs in Google Colab, so nothing is downloaded to your own computer. It fetches the complete assembly of every
lab-tested genome in cleaning v7 that we do not have yet (**64,850 genomes**, listed in `lab_genomes_todo.csv`)
from the BV-BRC API and writes each one **gzipped** to Google Drive:

`My Drive/FYP/genomes_lab_v7/<2 characters>/<genome_id>.fna.gz`

- **Size:** 247 GB of DNA, about **76 GB** gzipped. Check your Drive has about 85 GB free first
  (drive.google.com/settings/storage). The run checks this too and stops if there is not enough space.
- **Checked:** a genome is kept only if its length is within 1% of the length BV-BRC reports, the same rule as
  `experiments/genome/features/download_genomes.py`.
- **Resumable:** Colab stops after some hours. Open the notebook again and run all cells: genomes already on
  Drive are skipped. Keep this tab open while it runs.
- **Before you start:** put `lab_genomes_todo.csv` (from `experiments/genome/features/select_lab_genomes.py --todo`)
  in `My Drive/FYP/`. `genomes_lab_v7/lengths.csv` is optional; without it the first run fetches the lengths (a few minutes).

Genus order is round-robin, so a run stopped part way still holds a balanced sample.

## 1. Connect Google Drive
A window asks for permission to your Drive.

In [ ]:
# If this says "Mountpoint must not already contain files":
# Runtime → Disconnect and delete runtime, then Run all again.
from google.colab import drive
drive.mount('/content/drive')

## 2. Settings

In [ ]:
FOLDER = '/content/drive/MyDrive/FYP'
LIST = f'{FOLDER}/lab_genomes_todo.csv'   # which genomes to fetch
OUT = f'{FOLDER}/genomes_lab_v7'           # where they go
WORKERS = 8                                # parallel downloads; BV-BRC copes with 8
LIMIT = None                               # e.g. 20 for a quick test, None for all
CHECK_SPACE = True                         # set False only if the free-space figure is clearly wrong

## 3. The downloader
Same checks as the project's `download_genomes.py`, written for Drive. Run this cell; it only defines functions.

In [ ]:
import gzip
import hashlib
import json
import os
import shutil
import ssl
import time
import urllib.parse
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd

API = 'https://www.bv-brc.org/api'
USER_AGENT = 'amr-fyp-genome-download/1.0 (COMSATS FYP)'   # BV-BRC refuses Python's default
TOLERANCE = 0.01                                           # keep a file only within 1% of BV-BRC's length
RETRY_WAITS = [10, 20, 40, 60, 90, 120]
try:
    import certifi
    SSL_CONTEXT = ssl.create_default_context(cafile=certifi.where())
except ImportError:
    SSL_CONTEXT = ssl.create_default_context()


def get(url, accept, timeout=300):
    req = urllib.request.Request(url, headers={'Accept': accept, 'User-Agent': USER_AGENT})
    for wait in RETRY_WAITS + [None]:
        try:
            with urllib.request.urlopen(req, timeout=timeout, context=SSL_CONTEXT) as resp:
                return resp.read()
        except Exception as e:
            if wait is None:
                raise
            print(f'  retry in {wait}s after {e}')
            time.sleep(wait)


def shard_of(genome_id):
    """One of 256 subfolders, so no Drive folder holds more than a few hundred files."""
    return hashlib.md5(genome_id.encode()).hexdigest()[:2]


def path_of(out, genome_id):
    return os.path.join(out, shard_of(genome_id), f'{genome_id}.fna.gz')


def lengths(genome_ids, manifest_path, batch=200):
    """genome_length per genome from BV-BRC, saved to Drive so a restart skips this."""
    if os.path.exists(manifest_path):
        m = pd.read_csv(manifest_path, dtype={'genome_id': str})
        if set(genome_ids) <= set(m['genome_id']):
            return m
    rows = []
    for i in range(0, len(genome_ids), batch):
        ids = genome_ids[i:i + batch]
        q = (f'in(genome_id,({",".join(ids)}))'
             f'&select(genome_id,genome_length,contigs,genome_status)&limit({len(ids)})')
        rows += json.loads(get(f'{API}/genome/?{q}', 'application/json'))
        if (i // batch) % 25 == 0:
            print(f'  lengths {min(i + batch, len(genome_ids)):,} / {len(genome_ids):,}')
    m = pd.DataFrame(rows)
    m['genome_id'] = m['genome_id'].astype(str)
    m.to_csv(manifest_path, index=False)
    return m


def sequence_length(fasta):
    return sum(len(line.strip()) for line in fasta.splitlines() if not line.startswith(b'>'))


def fetch(out, genome_id, expected):
    """Download one assembly, check its length, write it gzipped. Returns bytes written."""
    q = urllib.parse.quote(f'eq(genome_id,{genome_id})', safe='(),') + '&limit(100000)'
    data = get(f'{API}/genome_sequence/?{q}', 'application/dna+fasta')
    got = sequence_length(data)
    if abs(got - expected) > TOLERANCE * expected:
        raise ValueError(f'got {got:,} bp, expected {expected:,}')
    packed = gzip.compress(data, compresslevel=6)
    if out.startswith('/content/drive') and not os.path.ismount('/content/drive'):
        # Drive disconnected: writing now would land on Colab's temporary disk
        raise SystemExit('Google Drive is no longer connected. Runtime → Disconnect and delete '
                         'runtime, then Run all: the download resumes from what is on Drive.')
    final = path_of(out, genome_id)
    os.makedirs(os.path.dirname(final), exist_ok=True)
    with open(final + '.part', 'wb') as fh:
        fh.write(packed)
    os.replace(final + '.part', final)     # a file exists only once it is complete
    return len(packed)


def existing(out):
    done = set()
    if os.path.isdir(out):
        for sub in os.listdir(out):
            d = os.path.join(out, sub)
            if os.path.isdir(d):
                done |= {f[:-len('.fna.gz')] for f in os.listdir(d) if f.endswith('.fna.gz')}
    return done


def run(list_csv, out, workers=8, limit=None, reserve_gb=5, check_space=True):
    os.makedirs(out, exist_ok=True)
    todo = pd.read_csv(list_csv, dtype={'genome_id': str})
    if limit:
        todo = todo.head(limit)
    meta = lengths(todo['genome_id'].tolist(), os.path.join(out, 'lengths.csv'))
    todo = todo.merge(meta[['genome_id', 'genome_length']], on='genome_id', how='left')
    unknown = todo[todo['genome_length'].isna()]
    done = existing(out)
    left = todo[todo['genome_length'].notna() & ~todo['genome_id'].isin(done)]

    # About 3.5 bytes of sequence per byte after gzip; check the space first
    need_gb = left['genome_length'].sum() / 3.5 / 1e9
    free_gb = shutil.disk_usage(out).free / 1e9
    print(f'{len(todo):,} genomes listed: {len(done):,} already on Drive, {len(left):,} to fetch, '
          f'{len(unknown):,} not on BV-BRC. Needs about {need_gb:.0f} GB; {free_gb:.0f} GB free.')
    if check_space and need_gb + reserve_gb > free_gb:
        print('Not enough space on Drive. Free some space or buy more storage, then run this again.')
        return

    failures = [{'genome_id': g, 'error': 'not found on BV-BRC'} for g in unknown['genome_id']]
    t0, written, n_ok = time.time(), 0, 0
    with ThreadPoolExecutor(workers) as pool:
        jobs = {pool.submit(fetch, out, r.genome_id, int(r.genome_length)): r.genome_id
                for r in left.itertuples()}
        for n, job in enumerate(as_completed(jobs), 1):
            try:
                written += job.result()
                n_ok += 1
            except SystemExit:
                pool.shutdown(wait=False, cancel_futures=True)
                raise
            except Exception as e:
                failures.append({'genome_id': jobs[job], 'error': str(e)})
                print(f'  FAILED {jobs[job]}: {e}')
            if n % 50 == 0 or n == len(jobs):
                rate = n / (time.time() - t0)
                print(f'  {n:,} / {len(jobs):,} done, {written / 1e9:.1f} GB written, '
                      f'about {(len(jobs) - n) / rate / 3600:.1f} h left')
    pd.DataFrame(failures, columns=['genome_id', 'error']).to_csv(
        os.path.join(out, 'failures.csv'), index=False)
    total = len(existing(out))
    print(f'Finished: {total:,} of {len(todo):,} genomes on Drive; {len(failures)} failed '
          f'(failures.csv). Run this cell again to retry them.')

## 4. Download
Prints progress every 50 genomes. If Colab disconnects, run all cells again to continue.

In [ ]:
run(LIST, OUT, workers=WORKERS, limit=LIMIT, check_space=CHECK_SPACE)

## 5. Check what is on Drive
Run any time, also from a new session after step 1.

In [ ]:
import glob, os
files = glob.glob(f'{OUT}/*/*.fna.gz')
size = sum(os.path.getsize(f) for f in files)
print(f'{len(files):,} genomes on Drive, {size / 1e9:.1f} GB')

## 6. Make sure every file reached Drive
Run this at the end of a session, before closing the tab.

In [ ]:
drive.flush_and_unmount()
print('All files written to Drive.')